#### Install

In [ ]:
# %pip install kaggle
# %pip uninstall keras

# %pip install h5py

%pip install tensorflow
# %pip install keras==2.1.5

# !kaggle kernels output harshavardhanbabu/keras-densenet-aptos -p ./assets/Aptos

#### Imports

In [1]:
import tensorflow as tf
import keras
import os

#### Paths

In [30]:
inception_model_path = "./assets/InceptionV3.Mader/full_retina_model.h5"
inception_weights_path = "./assets/InceptionV3.Mader/Resnet50_bestqwk.h5"
resnet_model_path = "./assets/ResNet50/full_retina_model (1).h5"
aptos_model_path = "./assets/Aptos/Resnet50_bestqwk.h5"

#### Model Init

In [28]:
import h5py

try:
    with h5py.File(inception_model_path, "r") as f:
        # Try accessing some data within the file
        print(list(f.keys()))  # Print the top-level keys/groups
except Exception as e:
    print(f"Error with h5py: {e}")

# ... Use the PyTorch model ...

['model_weights', 'optimizer_weights']


In [32]:
def top_2_accuracy(in_gt, in_pred):  # If you used this custom metric
    return top_k_categorical_accuracy(in_gt, in_pred, k=2)


model1 = keras.models.load_model(
    inception_model_path, custom_objects={"top_2_accuracy": top_2_accuracy}
)

EOFError: EOF read where object expected

In [19]:
import tensorflow as tf
from keras.applications import InceptionV3
from keras.models import Model
from keras.layers import (
    GlobalAveragePooling2D,
    Dropout,
    Dense,
    Input,
    Conv2D,
    Multiply,
    Reshape,
)

In [20]:
# Define the attention mechanism
def attention_block(inputs):
    # Compute channel-wise attention
    x = GlobalAveragePooling2D()(inputs)
    x = Reshape((1, 1, x.shape[-1]))(x)
    x = Conv2D(filters=x.shape[-1], kernel_size=1, activation="relu")(x)
    x = Conv2D(filters=x.shape[-1], kernel_size=1, activation="sigmoid")(x)
    return Multiply()([inputs, x])

In [21]:
# Define the complete model architecture
def create_model(input_shape=(512, 512, 3), num_classes=5):
    # Input layer
    inputs = Input(shape=input_shape)

    # Load the InceptionV3 model without the top layer
    base_model = InceptionV3(weights="imagenet", include_top=False, input_tensor=inputs)

    # Apply the attention mechanism to the output of the base model
    x = attention_block(base_model.output)

    # Add custom layers
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    # Create the full model
    model = Model(inputs=inputs, outputs=outputs)
    return model

In [22]:
# Initialize the model
model = create_model()

87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 25s 0us/step


In [23]:
model.load_weights(inception_weights_path)

print("Model loaded successfully with weights.")

ValueError: Layer count mismatch when loading weights from file. Model expected 191 layers, found 243 saved layers.

> **Archived / incomplete fragment below.** This cell references `tf_augmentor`, `flow_from_dataframe`,
> and other helpers that are never defined in this notebook (the source it was adapted from left them as
> `# ... remain unchanged ...` placeholders). The attention model defined above (cells labeled "Model Init")
> is complete and loads real saved weights; this trailing cell is a separate, unfinished retraining attempt
> kept for reference only.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from keras import backend as K
from keras.applications.inception_v3 import InceptionV3, preprocess_input
from keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout,
    Flatten,
    Input,
    Conv2D,
    multiply,
    LocallyConnected2D,
    Lambda,
    BatchNormalization,
)
from keras.models import Model, load_model
from keras.utils import to_categorical
from keras.metrics import top_k_categorical_accuracy
from keras.callbacks import (
    ModelCheckpoint,
    LearningRateScheduler,
    EarlyStopping,
    ReduceLROnPlateau,
)
from sklearn.model_selection import train_test_split
from tqdm import tqdm_notebook


# --- Data Loading and Preprocessing ---

base_image_dir = os.path.join(
    "..", "input", "diabetic-retinopathy-detection"
)  # Replace with your data path
retina_df = pd.read_csv(os.path.join(base_image_dir, "trainLabels.csv"))
retina_df["PatientId"] = retina_df["image"].map(lambda x: x.split("_")[0])
retina_df["path"] = retina_df["image"].map(
    lambda x: os.path.join(base_image_dir, "{}.jpeg".format(x))
)
retina_df["exists"] = retina_df["path"].map(os.path.exists)
retina_df["eye"] = retina_df["image"].map(
    lambda x: 1 if x.split("_")[-1] == "left" else 0
)
retina_df["level_cat"] = retina_df["level"].map(
    lambda x: to_categorical(x, 1 + retina_df["level"].max())
)

retina_df.dropna(inplace=True)
retina_df = retina_df[retina_df["exists"]]


# --- Data Splitting and Balancing ---

rr_df = retina_df[["PatientId", "level"]].drop_duplicates()
train_ids, valid_ids = train_test_split(
    rr_df["PatientId"], test_size=0.25, random_state=2018, stratify=rr_df["level"]
)
raw_train_df = retina_df[retina_df["PatientId"].isin(train_ids)]
valid_df = retina_df[retina_df["PatientId"].isin(valid_ids)]

train_df = (
    raw_train_df.groupby(["level", "eye"])
    .apply(lambda x: x.sample(75, replace=True))
    .reset_index(drop=True)
)  # Adjust sampling as needed


# --- Image Augmentation and Loading ---
IMG_SIZE = (512, 512)
batch_size = 48  # Adjust batch size as needed


# ... (tf_image_loader and tf_augmentor functions remain unchanged) ...

core_idg = tf_augmentor(
    out_size=IMG_SIZE,
    color_mode="rgb",
    vertical_flip=True,
    crop_probability=0.0,
    batch_size=batch_size,
)
valid_idg = tf_augmentor(
    out_size=IMG_SIZE,
    color_mode="rgb",
    crop_probability=0.0,
    horizontal_flip=False,
    vertical_flip=False,
    random_brightness=False,
    random_contrast=False,
    random_saturation=False,
    random_hue=False,
    rotation_range=0,
    batch_size=batch_size,
)

train_gen = flow_from_dataframe(core_idg, train_df, path_col="path", y_col="level_cat")
valid_gen = flow_from_dataframe(valid_idg, valid_df, path_col="path", y_col="level_cat")


# --- Model Definition ---
# If you have saved the model weights, load the architecture and then the weights:
try:
    retina_model = load_model(
        "full_retina_model.h5", custom_objects={"top_2_accuracy": top_2_accuracy}
    )
    print("Loaded model from disk")
    retina_model.load_weights("retina_weights.best.hdf5")  # Load your saved weights
    print("Loaded model weights from disk")

except OSError:  # Define the model architecture if loading fails.
    print("Model or weights not found on disk. Recreating model...")

    in_lay = Input(IMG_SIZE + (3,))
    base_pretrained_model = InceptionV3(
        input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet"
    )
    base_pretrained_model.trainable = False
    pt_depth = base_pretrained_model.get_output_shape_at(0)[-1]
    pt_features = base_pretrained_model(in_lay)

    bn_features = BatchNormalization()(pt_features)
    attn_layer = Conv2D(64, kernel_size=(1, 1), padding="same", activation="relu")(
        Dropout(0.5)(bn_features)
    )
    attn_layer = Conv2D(16, kernel_size=(1, 1), padding="same", activation="relu")(
        attn_layer
    )
    attn_layer = Conv2D(8, kernel_size=(1, 1), padding="same", activation="relu")(
        attn_layer
    )
    attn_layer = Conv2D(1, kernel_size=(1, 1), padding="valid", activation="sigmoid")(
        attn_layer
    )

    up_c2_w = np.ones((1, 1, 1, pt_depth))
    up_c2 = Conv2D(
        pt_depth,
        kernel_size=(1, 1),
        padding="same",
        activation="linear",
        use_bias=False,
        weights=[up_c2_w],
    )

    up_c2.trainable = False
    attn_layer = up_c2(attn_layer)

    mask_features = multiply([attn_layer, bn_features])
    gap_features = GlobalAveragePooling2D()(mask_features)
    gap_mask = GlobalAveragePooling2D()(attn_layer)

    gap = Lambda(lambda x: x[0] / x[1], name="RescaleGAP")([gap_features, gap_mask])
    gap_dr = Dropout(0.25)(gap)
    dr_steps = Dropout(0.25)(Dense(128, activation="relu")(gap_dr))
    out_layer = Dense(5, activation="softmax")(dr_steps)  # Output layer with 5 classes

    retina_model = Model(inputs=[in_lay], outputs=[out_layer])

    retina_model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["categorical_accuracy", top_2_accuracy],
    )

    retina_model.load_weights("retina_weights.best.hdf5")  # Load your saved weights


# ... (Rest of the code for training, evaluation, and attention visualization remains largely the same)